In [ ]:
import zipfile

with zipfile.ZipFile("/content/data.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/data")

print("Unzip done")

Unzip done


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

data_dir = "/content/data/data/train"

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(128, 128),
    batch_size=32
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(128, 128),
    batch_size=32
)

# Normalize
train_ds = train_ds.map(lambda x, y: (x / 255.0, y))
val_ds = val_ds.map(lambda x, y: (x / 255.0, y))

# Model
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(train_ds, validation_data=val_ds, epochs=5)

Found 2093 files belonging to 2 classes.
Using 1675 files for training.
Found 2093 files belonging to 2 classes.
Using 418 files for validation.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/5
53/53 ━━━━━━━━━━━━━━━━━━━━ 43s 683ms/step - accuracy: 0.8615 - loss: 0.3453 - val_accuracy: 0.9091 - val_loss: 0.2279
Epoch 2/5
53/53 ━━━━━━━━━━━━━━━━━━━━ 33s 542ms/step - accuracy: 0.9319 - loss: 0.1877 - val_accuracy: 0.9354 - val_loss: 0.1710
Epoch 3/5
53/53 ━━━━━━━━━━━━━━━━━━━━ 29s 546ms/step - accuracy: 0.9385 - loss: 0.1567 - val_accuracy: 0.9330 - val_loss: 0.1714
Epoch 4/5
53/53 ━━━━━━━━━━━━━━━━━━━━ 30s 563ms/step - accuracy: 0.9481 - loss: 0.1403 - val_accuracy: 0.9282 - val_loss: 0.1688
Epoch 5/5
53/53 ━━━━━━━━━━━━━━━━━━━━ 29s 551ms/step - accuracy: 0.9522 - loss: 0.1271 - val_accuracy: 0.9306 - val_loss: 0.1750


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("sunglasses_model.tflite", "wb") as f:
    f.write(tflite_model)

Saved artifact at '/tmp/tmpd4la0g2x'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  138526366499472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138526366502736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138526366502160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138526366501584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138526366503888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138526366502544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138526366501968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138526366509264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138526366504272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138526366502928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1385263665

In [ ]:
from google.colab import files
files.download("sunglasses_model.tflite")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>